# Características mais comuns de um domicílio brasileiro

**Pergunta:** Quais são as características mais comuns de um domicílio? (insumo para a documentação da smarthouse)

**Fonte:** Censo 2010 (Amostra) — grão de **domicílio**.

**Método (abordagem A — moda por atributo):** para cada atributo, o valor mais frequente *ponderado pelo peso amostral*. Não buscamos a combinação exata mais comum (B), que seria um arquétipo raro e enganoso.

Dois cuidados em toda conta:
- **Peso:** `SUM(peso_amostral)` = população estimada, não contagem de linhas da amostra.
- **Grão:** consultamos `domicilios` (1 linha = 1 domicílio), então não há duplo-contagem por morador.

**Nulos têm política por variável:** se o vazio é *não-resposta* (desconhecido), excluímos; se é *não-aplicável* (ex.: internet só perguntada a quem tem computador), o vazio é um "Não" informativo e entra no denominador.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# raiz do repo (onde está data/)
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

from lab.research.housing_reality.sources.census_2010 import _to_snake

GLOB = str(root / "data" / "census_2010" / "*" / "mapped" / "domicilios.parquet")
con = duckdb.connect()

# descrições do layout = rótulos autoritativos (contêm os códigos de cada categoria)
_layout = pd.read_csv(root / "src/lab/research/housing_reality/sources/census_2010_layout/domicilios.csv")
DESC = {(_to_snake(d) or str(v).lower()): d for v, d in zip(_layout["variable"], _layout["description"])}


def dist(coluna: str, incluir_nulos: bool = False) -> pd.DataFrame:
    """Distribuicao ponderada no grao de domicilio.

    incluir_nulos=False -> exclui o vazio do denominador (vazio = nao-resposta).
    incluir_nulos=True  -> mantem o vazio no denominador (vazio = nao-aplicavel = "Nao").
    """
    filtro = "" if incluir_nulos else f"WHERE {coluna} IS NOT NULL"
    return con.execute(f"""
        SELECT {coluna} AS valor,
               CAST(SUM(peso_amostral) AS BIGINT) AS domicilios,
               ROUND(100 * SUM(peso_amostral) / SUM(SUM(peso_amostral)) OVER (), 1) AS pct
        FROM read_parquet('{GLOB}')
        {filtro}
        GROUP BY 1 ORDER BY domicilios DESC
    """).df()

In [ ]:
total = con.execute(f"SELECT CAST(SUM(peso_amostral) AS BIGINT) FROM read_parquet('{GLOB}')").fetchone()[0]
print(f"Domicilios estimados (Brasil, 2010): {total:,}")

## Atributos numéricos (valor mais comum)

In [ ]:
numericas = {
    "quantas_pessoas_moravam_neste_domicilio_em_31_de_julho_de_2010": "moradores",
    "comodos_numero": "comodos",
    "comodos_como_dormitorio_numero": "dormitorios",
}
for col, nome in numericas.items():
    moda = dist(col).iloc[0]
    print(f"{nome:12} -> mais comum: {moda['valor']}  ({moda['pct']}% dos domicilios)")

pct_banheiro = con.execute(f"""
    SELECT ROUND(100.0 * SUM(peso_amostral) FILTER (WHERE banheiros_de_uso_exclusivo_numero >= 1)
                 / SUM(peso_amostral) FILTER (WHERE banheiros_de_uso_exclusivo_numero IS NOT NULL), 1)
    FROM read_parquet('{GLOB}')
""").fetchone()[0]
print(f"{'tem banheiro':12} -> {pct_banheiro}% dos domicilios")

## Atributos categóricos (perguntados a todos os domicílios)

Valores são **códigos**; a descrição do layout (impressa abaixo) traz o significado. Aqui o vazio é não-resposta (raro) → excluído.

In [ ]:
SITUACAO = {1: "Urbana", 2: "Rural"}
d = dist("situacao_do_domicilio")
d["valor"] = d["valor"].map(lambda v: SITUACAO.get(v, v))
print("Situacao urbana/rural:")
print(d.to_string(index=False))

In [ ]:
for col in [
    "abastecimento_de_agua_forma",
    "esgotamento_sanitario_tipo",
    "domicilio_condicao_de_ocupacao",
    "geladeira_existencia",
]:
    print("=" * 70)
    print(col)
    print("layout:", DESC.get(col, "(nao encontrado)"))
    print(dist(col).head(6).to_string(index=False))
    print()

## Variáveis condicionais (cuidado!)

Algumas perguntas só são feitas a um subconjunto, e o vazio significa **"não-aplicável"**, não "não respondeu". Excluir o vazio aqui responde a pergunta errada.

- **Internet** (`microcomputador_com_acesso_a_internet_existencia`): só perguntada a quem **tem computador**. Vazio = sem computador = sem internet → conta como **Não**, denominador = todos os domicílios.
- **Aluguel** (`valor_do_aluguel_em_reais`): só existe para domicílios **alugados** — mesma pegadinha, tratar à parte.

In [ ]:
def rotulo_internet(v):
    if v == 1:
        return "Sim"
    if v == 2:
        return "Nao (tem PC, sem internet)"
    return "Nao (sem computador)"

d = dist("microcomputador_com_acesso_a_internet_existencia", incluir_nulos=True)
d["valor"] = d["valor"].map(rotulo_internet)
print("Internet (condicional — vazio = sem computador = Nao):")
print(d.to_string(index=False))

pct_internet = float(d.loc[d["valor"] == "Sim", "pct"].iloc[0])
print(f"\n-> % de domicilios com internet (denominador = todos): {pct_internet}%")

## Próximos passos
- Rótulos limpos dos categóricos (água, esgoto, ocupação...) → enums da Fase 2.
- Auditar quais outras variáveis são condicionais antes de usá-las.
- Mesmo retrato no PNADC 2025 (precisa mapear `id_domicilio` do PNADC para colapsar ao grão de domicílio) → comparação 2010 × 2025.